# Layer Contribution Ranking

Rank layers by their impact: logit contribution, norm, direction alignment,
entropy change, and cumulative effect.

## Why This Matters

Layer contribution ranking characterizes what each layer contributes to the model's computation. Understanding layer-level organization — which layers are critical, which are redundant, and how they specialize — is essential for both interpretability and efficient model design.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.layer_contribution_ranking import (
    layer_logit_contribution, layer_norm_contribution,
    layer_direction_importance, layer_entropy_contribution,
    layer_cumulative_effect,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Logit Contribution

Which layers contribute most to the predicted token's logit?

In [ ]:
result = layer_logit_contribution(model, tokens)
print(f"Target token: {result['target_token']}\n")
for p in result['per_layer']:
    bar = '█' * int(abs(p['total_logit_contrib']) * 20)
    sign = '+' if p['total_logit_contrib'] > 0 else '-'
    print(f"  [{sign}] Layer {p['layer']}: attn={p['attn_logit_contrib']:+.4f}, "
          f"mlp={p['mlp_logit_contrib']:+.4f}, total={p['total_logit_contrib']:+.4f} {bar}")

## Norm Contribution

Which layers modify the residual stream the most?

In [ ]:
result = layer_norm_contribution(model, tokens)
for p in result['per_layer']:
    bar = '█' * int(p['total_norm'] * 10)
    print(f"  Layer {p['layer']}: attn={p['attn_norm']:.4f}, "
          f"mlp={p['mlp_norm']:.4f}, total={p['total_norm']:.4f} {bar}")

## Direction Importance

Which layers push the residual toward its final direction?

In [ ]:
result = layer_direction_importance(model, tokens)
print(f"Constructive: {result['n_constructive']}/4\n")
for p in result['per_layer']:
    sign = '+' if p['is_constructive'] else '-'
    print(f"  [{sign}] Layer {p['layer']}: proj={p['projection']:+.4f}, "
          f"cos={p['cosine_with_final']:+.4f}")

## Entropy Contribution

Do layers sharpen or broaden the prediction?

In [ ]:
result = layer_entropy_contribution(model, tokens)
print(f"Sharpening layers: {result['n_sharpening']}/4\n")
for p in result['per_layer']:
    arrow = '↓' if p['sharpens'] else '↑'
    print(f"  Layer {p['layer']}: {p['pre_entropy']:.4f} → {p['post_entropy']:.4f} "
          f"(Δ={p['entropy_change']:+.4f}) [{arrow}]")

## Cumulative Effect

How does the prediction build up through layers?

In [ ]:
result = layer_cumulative_effect(model, tokens)
print(f"Target token: {result['target_token']}\n")
for c in result['cumulative']:
    top = '✓' if c['is_top1'] else '✗'
    print(f"  Layer {c['layer']}: logit={c['target_logit']:+.4f}, "
          f"rank={c['target_rank']} [{top}]")